<a href="https://colab.research.google.com/github/Vishnu0920/Recommender_System_Models/blob/main/AdverserialCollaborativeFIlteringWithAutoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.model_selection import train_test_split

# Load the dataset
data = pd.read_csv('Grade.csv')

# Normalize grades to the range [0, 1]
scaler = MinMaxScaler()
data['normalized_grade'] = scaler.fit_transform(data[['course_grade']])

# Handle duplicates by averaging grades for the same student and course
data = data.groupby(['student_id', 'course_id']).agg({'normalized_grade': 'mean'}).reset_index()

# Create user-item interaction matrix
user_item_matrix = data.pivot(index='student_id', columns='course_id', values='normalized_grade').fillna(0)

# Split data into training and testing sets
train_data, test_data = train_test_split(user_item_matrix, test_size=0.2, random_state=42)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Define the model
input_dim = user_item_matrix.shape[1]
hidden_dim = 64
autoencoder = Autoencoder(input_dim, hidden_dim)

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(Discriminator, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# Define the discriminator
discriminator = Discriminator(input_dim, hidden_dim)
discriminator_criterion = nn.BCELoss()
discriminator_optimizer = optim.Adam(discriminator.parameters(), lr=0.001)


In [ ]:
# Prepare data loaders
train_loader = DataLoader(TensorDataset(torch.tensor(train_data.values).float()), batch_size=32, shuffle=True)

# Training loop
num_epochs = 50

for epoch in range(num_epochs):
    autoencoder.train()
    discriminator.train()

    for data in train_loader:
        inputs = data[0]

        # Train Autoencoder
        optimizer.zero_grad()
        reconstructed = autoencoder(inputs)
        loss = criterion(reconstructed, inputs)
        loss.backward()
        optimizer.step()

        # Train Discriminator
        discriminator_optimizer.zero_grad()
        real_labels = torch.ones(inputs.size(0), 1)
        fake_labels = torch.zeros(inputs.size(0), 1)

        real_outputs = discriminator(inputs)
        fake_outputs = discriminator(reconstructed.detach())

        d_loss_real = discriminator_criterion(real_outputs, real_labels)
        d_loss_fake = discriminator_criterion(fake_outputs, fake_labels)
        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        discriminator_optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, D_Loss: {d_loss.item():.4f}')


Epoch [1/50], Loss: 0.0842, D_Loss: 0.3884
Epoch [2/50], Loss: 0.0414, D_Loss: 1.0506
Epoch [3/50], Loss: 0.0386, D_Loss: 1.2398
Epoch [4/50], Loss: 0.0337, D_Loss: 0.9632
Epoch [5/50], Loss: 0.0357, D_Loss: 0.8464
Epoch [6/50], Loss: 0.0398, D_Loss: 0.6271
Epoch [7/50], Loss: 0.0305, D_Loss: 0.5784
Epoch [8/50], Loss: 0.0270, D_Loss: 0.5669
Epoch [9/50], Loss: 0.0229, D_Loss: 0.5476
Epoch [10/50], Loss: 0.0260, D_Loss: 0.4094
Epoch [11/50], Loss: 0.0230, D_Loss: 0.3460
Epoch [12/50], Loss: 0.0186, D_Loss: 0.4301
Epoch [13/50], Loss: 0.0212, D_Loss: 0.3575
Epoch [14/50], Loss: 0.0217, D_Loss: 0.3688
Epoch [15/50], Loss: 0.0154, D_Loss: 0.4232
Epoch [16/50], Loss: 0.0181, D_Loss: 0.3522
Epoch [17/50], Loss: 0.0185, D_Loss: 0.2915
Epoch [18/50], Loss: 0.0159, D_Loss: 0.3977
Epoch [19/50], Loss: 0.0152, D_Loss: 0.4564
Epoch [20/50], Loss: 0.0145, D_Loss: 0.4384
Epoch [21/50], Loss: 0.0142, D_Loss: 0.3817
Epoch [22/50], Loss: 0.0144, D_Loss: 0.4127
Epoch [23/50], Loss: 0.0142, D_Loss: 0.47

In [ ]:
def get_recommendations_from_input(course_ids, course_grades, top_n=5):
    autoencoder.eval()

    # Normalize the input grades to the range [0, 1]
    scaler = MinMaxScaler()
    normalized_grades = scaler.fit_transform(np.array(course_grades).reshape(-1, 1)).flatten()

    # Create an empty user vector
    user_vector = np.zeros(user_item_matrix.shape[1])

    # Map the input course_ids to the corresponding indices in the user_item_matrix
    for course_id, grade in zip(course_ids, normalized_grades):
        if course_id in user_item_matrix.columns:
            user_vector[user_item_matrix.columns.get_loc(course_id)] = grade

    user_vector = torch.tensor(user_vector).float().unsqueeze(0)

    with torch.no_grad():
        reconstructed_vector = autoencoder(user_vector).numpy().flatten()

    # Preserve the original input course IDs
    input_course_ids = course_ids.copy()

    course_ids = user_item_matrix.columns
    recommendations = sorted(zip(course_ids, reconstructed_vector), key=lambda x: x[1], reverse=True)

    # Filter out courses that were in the input
    recommendations = [course[0] for course in recommendations if course[0] not in input_course_ids]

    return recommendations[:top_n]

# Example usage
input_course_ids = ['CHEMISTRY LABORATORY', 'GENERAL CHEMISTRY', 'ELECTRICAL SCIENCES']  # Replace with actual course IDs
input_course_grades = [8, 6, 9]  # Replace with actual grades from 0 to 10

recommendations = get_recommendations_from_input(input_course_ids, input_course_grades)
print(f'Top {len(recommendations)} course recommendations based on input: {recommendations}')


Top 5 course recommendations based on input: ['MATHEMATICS I', 'TECHNICAL REPORT WRITING', 'ENGINEERING GRAPHICS', 'BIOLOGY LABORATORY', 'GENERAL BIOLOGY']
